In [1]:
!pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.0/476.0 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.2 MB/s eta 0:00:00


In [12]:
from bs4 import BeautifulSoup

def extract_stock_data(html):
  """
  HTMLから株価データを抽出する関数

  Args:
    html: 解析対象のHTML

  Returns:
    list: 日付、始値、高値、安値、終値のリスト
  """
  soup = BeautifulSoup(html, 'html.parser')
  #print(soup)
  #date_element = soup.select_one('.chart-nav-wrap .date')
  #highcharts-0 > div.highcharts-tooltip
  #highcharts-0 > div.highcharts-tooltip > span > table > tbody > tr:nth-child(1) > td:nth-child(1)
  #//*[@id="highcharts-0"]/div[1]/span/table/tbody/tr[1]/td[1]
  date_element = soup.find(xpath='//*[@id="highcharts-0"]/div[1]/span/table/tbody/tr[1]/td[1]')
  print(f"date_element:{date_element}")
  date_str = date_element.text.strip() if date_element else None
  print(f"date_str:{date_str}")

  values_element = soup.select('.chart-nav-wrap .val')
  values = [value.text.strip() for value in values_element] if values_element else []

  if date_str and len(values) == 4:
    year, month, day = map(int, date_str.split('/'))
    date_str = f"{year:04d}-{month:02d}-{day:02d}"
    return [date_str] + values
  else:
    return None

In [13]:
from selenium import webdriver
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def get_stock_values(driver, url):
  """
  Webページから株価データを抽出する関数

  Args:
    driver: Selenium WebDriverオブジェクト
    url: アクセスするURL

  Returns:
    list: 株価データのリスト
  """
  driver.get(url)
  wait = WebDriverWait(driver, 10)

  # グラフの要素を取得
  #chart_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '.highcharts-area')))
  chart_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '.highcharts-background')))

  # グラフの中央にマウスを移動
  actions = ActionChains(driver)
  actions.move_to_element(chart_element).perform()
  width = chart_element.size['width']
  print(width)
  actions.move_by_offset(width // 2, 0).perform()

  stock_data = []
  # 1pxずつ左にマウスをずらしながらデータを取得
  for i in range(width // 2):
    print(i)
    actions.move_by_offset(-1, 0).perform()
    #print(driver.page_source)
    stock_data.append(extract_stock_data(driver.page_source))

  return stock_data

In [14]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options


# WebDriverの設定と起動
options = webdriver.ChromeOptions()
options.add_argument('--headless')  # ヘッドレスモードで実行
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')  # GPU を無効化
options.add_argument('--remote-debugging-port=9222')  # デバッグポートを指定

driver = webdriver.Chrome(options=options)

# スクレイピング開始時間
start_time = time.time()

url = 'https://www.nikkei.com/markets/worldidx/chart/nk225/?type=6month'
stock_data = get_stock_values(driver, url)

print(stock_data)

# スクレイピング終了時間
end_time = time.time()

# スクレイピングにかかった時間を表示
elapsed_time = end_time - start_time
print(f"スクレイピングにかかった時間: {elapsed_time:.2f}秒")

# 取得したデータを整形して表示
for data in stock_data:
  if data:
    print(data)

# ブラウザを閉じる
driver.quit()

650
0
date_element:None
date_str:None
1
date_element:None
date_str:None
2
date_element:None
date_str:None
3
date_element:None
date_str:None
4
date_element:None
date_str:None
5
date_element:None
date_str:None
6
date_element:None
date_str:None
7
date_element:None
date_str:None
8
date_element:None
date_str:None
9


KeyboardInterrupt: 